In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional

Xpath= r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\X.npy"
Ypath= r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\y.npy"
modelpath = r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\classification_file\cnn\250821_125241\model.pth"

X = np.load(Xpath)
Y = np.load(Ypath)
print(X.shape)
print(Y.shape)


(4201, 2000, 2)
(4201,)


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append(r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\classification_file\cnn\250821_125241")
from network import Network


# モデルとデータのロード
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Network()
model.load_state_dict(torch.load(modelpath, map_location=device))
model.eval()
model.to(device)

# 可視化したいサンプルを選択
sample_idx = 0
x = X[sample_idx]  # shape: (時系列長さ, 特徴数)
x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 時系列長さ, 特徴数)

# Grad-CAM用のフック
activations = []
grads = []

def forward_hook(module, input, output):
    activations.append(output.detach())

def backward_hook(module, grad_in, grad_out):
    grads.append(grad_out[0].detach())

# 最終Conv1d層を取得
for m in reversed(model.model):
    if isinstance(m, torch.nn.Conv1d):
        target_layer = m
        break

# フック登録
forward_handle = target_layer.register_forward_hook(forward_hook)
backward_handle = target_layer.register_backward_hook(backward_hook)

# 推論と勾配計算
x_tensor = x_tensor.permute(0,2,1)  # (1, 特徴数, 時系列長さ)
output = model.model(x_tensor)
pred_class = output.argmax(dim=1).item()

model.zero_grad()
score = output[0, pred_class]
score.backward()

# Grad-CAM計算
activation = activations[0][0]  # (チャネル, 時系列長さ)
gradient = grads[0][0]          # (チャネル, 時系列長さ)
weights = gradient.mean(dim=1)  # (チャネル,)

cam = (weights[:, None] * activation).sum(dim=0)  # (時系列長さ,)
cam = F.relu(cam)
cam = cam.cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # 0-1正規化

# フック解除
forward_handle.remove()
backward_handle.remove()

# 可視化
plt.figure(figsize=(12,4))
plt.plot(x[:,0], label='波形(電流など)')
plt.twinx()
plt.plot(cam, color='red', alpha=0.5, label='Grad-CAM')
plt.title(f"Grad-CAM 可視化 (予測クラス: {pred_class})")
plt.legend(loc='upper right')
plt.show()

ModuleNotFoundError: No module named 'network'

In [ ]:
import sys
sys.path.append(r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\classification_file\cnn\250821_125241")
from network import Network

In [ ]:
import sys
sys.path.append(r"C:\Users\ylab\Downloads\250327_新型チップ測定データ\250327_新型チップ測定データ\中間発表用解析\250821_123701\classification_file\cnn\250821_125241")
from network import Network